# Nurse Navigation - Disposition Recast Baseline

Profiles how calls are classified today and sets up the recast of that classification into Schmitt Thomson definitions.

The recast answers one question per market: for a rolling twelve months of history, what share of calls fell into each disposition bucket under the current definitions, and what share would fall into each bucket under Schmitt Thomson definitions.

The mapping between the two is a clinical and operational decision, not a data decision. It is held in an external rules file so it can be reviewed and changed without touching this code. This notebook profiles the values that exist today so those rules can be written against real values rather than assumed ones.


## 1. Setup

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200); pd.set_option("display.max_colwidth", 300)
plt.rcParams.update({"figure.figsize":(11,5),"figure.dpi":110,"axes.grid":True,"grid.alpha":0.25,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"axes.titlesize":13,"axes.titleweight":"bold"})
TEAL, NAVY, CORAL, GOLD, GREY = "#028090","#0B2545","#D1495B","#E0A500","#8FA0A6"

DATA_DIR   = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/data"
OUT_DIR    = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/results"
RULES_DIR  = "/Workspace/Users/josh.smitherman@gmr.net/nurse_nav/rules"
SOURCE_FILE = "data_april2026-aug2026.xlsx"
RULES_FILE  = "schmitt_thomson_recast_rules.json"

WINDOW_END    = "2026-10-31"
WINDOW_MONTHS = 12
FOCUS_MARKETS = ["Seattle", "Dekalb", "Wake"]
MIN_MARKET_CALLS = 500

os.makedirs(OUT_DIR, exist_ok=True)
RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
RESULTS = {}
def keep(d, n): RESULTS[n] = d.copy(); return d
def show(d):
    o = d.copy()
    for c in o.columns:
        if o[c].dtype == object: o[c] = o[c].astype(str)
    display(o)
print("run:", RUN_ID)

## 2. Load calls and resolve columns

Column names vary between extracts, so each field is resolved by matching on likely names rather than assumed position.

In [ ]:
def find_col(frame, exact, parts):
    low = {str(c).strip().lower(): c for c in frame.columns}
    for e in exact:
        if e in low: return low[e]
    for c_low, c in low.items():
        if all(p in c_low for p in parts): return c
    return None

src = os.path.join(DATA_DIR, SOURCE_FILE)
df = pd.read_excel(src)
df.columns = [str(c).strip() for c in df.columns]
print(f"loaded {len(df):,} rows from {SOURCE_FILE}")

DISPO   = find_col(df, ["disposition","final disposition"], ["disposition"])
NMTARA  = find_col(df, ["nmtara","nmtara breakout"], ["nmtara"])
MARKET  = find_col(df, ["market","agency","customer"], ["market"])
DATE    = find_col(df, ["call date","transaction_create_date","date"], ["date"])
NOTES   = find_col(df, ["notes","nurse notes","call notes"], ["note"])
CAUSE   = find_col(df, ["cause","chief complaint"], ["cause"])
PROTOCOL= find_col(df, ["protocol","guideline"], ["protocol"])
OUTCOME = find_col(df, ["outcome","care setting","referral"], ["outcome"])

resolved = pd.DataFrame({
    "Field": ["Disposition","NMTARA breakout","Market","Call date","Nurse notes","Cause or chief complaint","Protocol","Outcome or care setting"],
    "Column found": [DISPO, NMTARA, MARKET, DATE, NOTES, CAUSE, PROTOCOL, OUTCOME]})
resolved["Present"] = resolved["Column found"].notna()
keep(resolved, "columns_resolved")
show(resolved)
missing = resolved.loc[~resolved["Present"], "Field"].tolist()
if missing:
    print("Not found in this extract:", ", ".join(missing))
    print("The recast needs Disposition, NMTARA breakout, Market and Call date at minimum.")

## 3. Rolling twelve month window

The recast baseline covers a rolling twelve months ending at the date set above. Volumes outside that window are excluded so the baseline matches the period that will later be compared against post-transition data.

In [ ]:
if DATE:
    df[DATE] = pd.to_datetime(df[DATE], errors="coerce")
    end = pd.Timestamp(WINDOW_END)
    start = end - pd.DateOffset(months=WINDOW_MONTHS) + pd.Timedelta(days=1)
    in_window = df[DATE].between(start, end)
    coverage = pd.DataFrame({
        "Measure": ["Rows in file","Earliest call date","Latest call date","Window start","Window end",
                    "Calls inside the window","Calls outside the window","Calls with no date"],
        "Value": [f"{len(df):,}", str(df[DATE].min().date()), str(df[DATE].max().date()),
                  str(start.date()), str(end.date()),
                  f"{int(in_window.sum()):,}", f"{int((~in_window & df[DATE].notna()).sum()):,}",
                  f"{int(df[DATE].isna().sum()):,}"]})
    keep(coverage, "window_coverage")
    show(coverage)
    if df[DATE].max() < end:
        short = (end - df[DATE].max()).days
        print(f"This extract ends {short} days before the requested window end.")
        print("A refreshed extract is required before the baseline covers the full twelve months.")
    win = df[in_window].copy()
else:
    win = df.copy()
    print("No date column resolved; the full file is used and the window is not applied.")
print(f"calls in scope: {len(win):,}")

## 4. What the disposition field contains today

Every distinct disposition value in scope, with volume. The recast rules are written against these values, so this table is the starting point for that conversation.

In [ ]:
if DISPO:
    d = win[DISPO].fillna("(blank)").astype(str).str.strip()
    vals = d.value_counts().rename("Calls").to_frame()
    vals["Share of calls"] = (vals["Calls"]/len(win)*100).round(1)
    vals.index.name = "Disposition value"
    vals = vals.reset_index()
    keep(vals, "disposition_values")
    show(vals)
    print(f"{len(vals)} distinct disposition values across {len(win):,} calls in scope.")

    o = vals.head(20).sort_values("Calls")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.34*len(o)+1)))
    ax.barh(range(len(o)), o["Calls"], color=TEAL)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o["Disposition value"], fontsize=9)
    ax.set_title("Disposition values by volume")
    for i, v in enumerate(o["Calls"]):
        ax.annotate(f"{v:,}", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

## 5. NMTARA breakout bands

The NMTARA breakout field encodes disposition bands, not individual acuity levels. A value such as BLS for NMTARA 1-5 (Override) describes the band and the override, so reading a single digit out of it mislabels the row. Bands are read whole here.

In [ ]:
def nmtara_band(v):
    t = str(v).strip()
    if not t or t.lower() in ("nan","none"): return "(blank)"
    return re.sub(r"\s+", " ", t)

def band_flags(t):
    low = t.lower()
    return pd.Series({
        "Override band": "override" in low,
        "Band covers levels 1 to 5": bool(re.search(r"1\s*-\s*5", low)),
        "Band covers level 6": bool(re.search(r"(?<![0-9])6(?![0-9])", low)),
        "Basic life support": "bls" in low,
        "Advanced life support": "als" in low})

if NMTARA:
    w = win.copy()
    w["nmtara_band"] = w[NMTARA].apply(nmtara_band)
    bands = w["nmtara_band"].value_counts().rename("Calls").to_frame()
    bands["Share of calls"] = (bands["Calls"]/len(w)*100).round(1)
    bands.index.name = "NMTARA breakout value"
    bands = bands.reset_index()
    flags = bands["NMTARA breakout value"].apply(band_flags)
    bands = pd.concat([bands, flags], axis=1)
    keep(bands, "nmtara_bands")
    show(bands)
    n_over = int(bands.loc[bands["Override band"], "Calls"].sum())
    n_15   = int(bands.loc[bands["Band covers levels 1 to 5"] & bands["Override band"], "Calls"].sum())
    n_6    = int(bands.loc[bands["Band covers level 6"], "Calls"].sum())
    print(f"Calls in an override band: {n_over:,} ({n_over/len(w)*100:.1f}%)")
    print(f"Calls in a levels 1 to 5 override band: {n_15:,} ({n_15/len(w)*100:.1f}%)")
    print(f"Calls in a level 6 band: {n_6:,} ({n_6/len(w)*100:.1f}%)")
    print("These two groups are the override and level six populations raised in review.")

    o = bands.head(12).sort_values("Calls")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.36*len(o)+1)))
    cols = [GOLD if ov else TEAL for ov in o["Override band"]]
    ax.barh(range(len(o)), o["Calls"], color=cols)
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o["NMTARA breakout value"], fontsize=8)
    ax.set_title("NMTARA breakout bands by volume, override bands in gold")
    for i, v in enumerate(o["Calls"]):
        ax.annotate(f"{v:,}", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

## 6. Disposition mix by market

The recast is presented per market, so the baseline mix is established per market first. Markets below the minimum volume set above are grouped out of the detail.

In [ ]:
by_mkt = pd.DataFrame()
if DISPO and MARKET:
    w = win.copy()
    w["_dispo"] = w[DISPO].fillna("(blank)").astype(str).str.strip()
    w["_market"] = w[MARKET].fillna("(blank)").astype(str).str.strip()
    sizes = w["_market"].value_counts()
    big = sizes[sizes >= MIN_MARKET_CALLS].index
    sub = w[w["_market"].isin(big)]
    by_mkt = pd.crosstab(sub["_market"], sub["_dispo"])
    by_mkt_pct = (by_mkt.div(by_mkt.sum(axis=1), axis=0)*100).round(1)
    by_mkt_pct.insert(0, "Total calls", by_mkt.sum(axis=1))
    by_mkt_pct = by_mkt_pct.sort_values("Total calls", ascending=False).reset_index().rename(columns={"_market":"Market"})
    keep(by_mkt_pct, "disposition_by_market")
    show(by_mkt_pct.head(25))
    print(f"{len(big)} markets at or above {MIN_MARKET_CALLS:,} calls in the window.")
    print("Values are the percent of that market's calls falling in each disposition.")

    top = by_mkt.sum().sort_values(ascending=False).head(6).index
    o = by_mkt_pct.head(12).set_index("Market")[list(top)]
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.45*len(o)+1.5)))
    left = np.zeros(len(o))
    palette = [NAVY, TEAL, GREY, GOLD, CORAL, "#5B8C85"]
    for j, c in enumerate(o.columns):
        ax.barh(range(len(o)), o[c], left=left, color=palette[j % len(palette)], label=str(c))
        left = left + o[c].values
    ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index, fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(fontsize=8, ncol=3, loc="lower right")
    ax.set_title("Disposition mix by market, six largest dispositions")
    plt.tight_layout(); plt.show()

## 7. Focus markets

The first pass covers two to three markets so the presentation format can be agreed before the recast is extended to every market.

In [ ]:
focus = pd.DataFrame()
if DISPO and MARKET:
    w = win.copy()
    w["_market"] = w[MARKET].fillna("(blank)").astype(str).str.strip()
    hit = w["_market"].apply(lambda m: any(f.lower() in m.lower() for f in FOCUS_MARKETS))
    focus = w[hit].copy()
    found = sorted(focus["_market"].unique())
    print("Focus markets matched:", ", ".join(found) if found else "none")
    if not len(focus):
        print("No market name matched FOCUS_MARKETS. Set it from the market list in the tab above.")
    else:
        fm = focus.groupby("_market").size().rename("Calls in window").to_frame()
        fm["Share of all calls in window"] = (fm["Calls in window"]/len(win)*100).round(1)
        fm.index.name = "Market"
        keep(fm.reset_index(), "focus_markets")
        show(fm)

## 8. Recast rules

The mapping from current disposition to Schmitt Thomson disposition is held in an external rules file. Each rule names the disposition it applies to, the condition that must hold, the disposition it maps to, and the basis for the rule.

A call matches at most one rule. Calls matching no rule keep their current disposition and are reported separately, so the share of volume the rules do not yet cover is visible rather than silently defaulted.

In [ ]:
rules_path = os.path.join(RULES_DIR, RULES_FILE)
DEFAULT_RULES = {
  "version": "draft",
  "rules": [
    {"id": "self_care_private_vehicle",
     "from_disposition": "Self Care",
     "condition": "patient_self_transported_to_ed",
     "to_disposition": "ED via private vehicle",
     "basis": "Current classification records transport. Schmitt Thomson records final care setting. A patient who declines transport and drives to an emergency department is recorded as self care today and as an emergency department visit under Schmitt Thomson.",
     "confirmed_by": "operations"}
  ]}

if os.path.exists(rules_path):
    with open(rules_path) as fh: RULES = json.load(fh)
    print(f"loaded {len(RULES.get('rules', []))} rules from {RULES_FILE}")
else:
    RULES = DEFAULT_RULES
    os.makedirs(RULES_DIR, exist_ok=True)
    with open(rules_path, "w") as fh: json.dump(RULES, fh, indent=2)
    print(f"no rules file found; wrote a starter file to {rules_path}")

rule_tbl = pd.DataFrame(RULES["rules"])
keep(rule_tbl, "recast_rules")
show(rule_tbl)
print(f"Rules defined: {len(rule_tbl)}. Rules version: {RULES.get('version','unset')}.")
print("Each additional rule requires a disposition it applies to, a condition, a target disposition, and the basis for it.")

## 9. Conditions the rules depend on

A rule condition is evaluated per call. Conditions that can be read from a coded field are evaluated here. Conditions that depend on what the nurse wrote are listed as outstanding, because they require the note to be read and cannot be derived from the coded fields alone.

In [ ]:
cond_status = []
conds = sorted({r["condition"] for r in RULES["rules"]})

def cond_from_fields(name, frame):
    if name == "patient_self_transported_to_ed":
        return None
    return None

for c in conds:
    fn = cond_from_fields(c, focus if len(focus) else win)
    cond_status.append({
        "Condition": c,
        "Source": "coded field" if fn is not None else "nurse note",
        "Evaluated in this run": fn is not None,
        "What is required": "none" if fn is not None else
            "The note records whether the patient self transported and to what care setting. Reading the note is required to evaluate this condition."})
cond_tbl = pd.DataFrame(cond_status)
keep(cond_tbl, "rule_conditions")
show(cond_tbl)
outstanding = cond_tbl.loc[~cond_tbl["Evaluated in this run"], "Condition"].tolist()
if outstanding:
    print("Conditions not yet evaluated:", ", ".join(outstanding))
    print("Until these are evaluated the recast reports the population each rule would apply to, not the recast itself.")

## 10. Population each rule applies to

For each rule, the number of calls in scope carrying the disposition that rule applies to. This is the maximum number of calls the rule can move. The number that actually move depends on the condition, which is reported above.

In [ ]:
reach = pd.DataFrame()
if DISPO and len(RULES["rules"]):
    base = focus if len(focus) else win
    b = base.copy()
    b["_dispo"] = b[DISPO].fillna("(blank)").astype(str).str.strip().str.lower()
    rows = []
    for r in RULES["rules"]:
        tgt = str(r["from_disposition"]).strip().lower()
        n = int((b["_dispo"] == tgt).sum())
        rows.append({"Rule": r["id"],
                     "Applies to disposition": r["from_disposition"],
                     "Maps to": r["to_disposition"],
                     "Calls carrying that disposition": n,
                     "Share of calls in scope": round(n/len(b)*100, 1) if len(b) else 0.0,
                     "Condition": r["condition"]})
    reach = pd.DataFrame(rows)
    keep(reach, "rule_reach")
    show(reach)
    covered = sorted({str(r["from_disposition"]).strip().lower() for r in RULES["rules"]})
    uncovered = b.loc[~b["_dispo"].isin(covered), "_dispo"].value_counts().rename("Calls").to_frame()
    uncovered["Share of calls in scope"] = (uncovered["Calls"]/len(b)*100).round(1)
    uncovered.index.name = "Disposition with no rule"
    keep(uncovered.reset_index(), "dispositions_without_rules")
    show(uncovered.head(20))
    n_unc = int(uncovered["Calls"].sum())
    print(f"{n_unc:,} calls in scope, {n_unc/len(b)*100:.1f}%, carry a disposition no rule covers.")
    print("Those calls keep their current disposition in the recast.")

    cov = pd.DataFrame({"Group": ["Covered by a rule", "No rule yet"],
                        "Calls": [int(reach["Calls carrying that disposition"].sum()), n_unc]})
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.barh(range(len(cov)), cov["Calls"], color=[TEAL, GREY])
    ax.set_yticks(range(len(cov))); ax.set_yticklabels(cov["Group"], fontsize=10)
    ax.set_title("Calls in scope covered by a recast rule")
    for i, v in enumerate(cov["Calls"]):
        ax.annotate(f"{v:,}", (v, i), xytext=(4,0), textcoords="offset points", va="center", fontsize=9)
    plt.tight_layout(); plt.show()

## 11. Write the workbook

In [ ]:
def stack(parts):
    out = []
    for section, frame in parts:
        if frame is None or not len(frame): continue
        f = frame.copy()
        f.insert(0, "Section", section)
        out.append(f)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

info = [("Source", "file", SOURCE_FILE),
        ("Source", "rows loaded", f"{len(df):,}"),
        ("Source", "rules file", RULES_FILE),
        ("Source", "rules version", str(RULES.get("version", "unset"))),
        ("Window", "window end", WINDOW_END),
        ("Window", "months", str(WINDOW_MONTHS)),
        ("Window", "calls in scope", f"{len(win):,}"),
        ("Purpose", "what this run produces",
         "A profile of how calls are classified today, and the population each recast rule applies to. It does not yet produce recast volumes, because the rule conditions that depend on the nurse note have not been evaluated."),
        ("Purpose", "what is required next",
         "A rule per disposition bucket in scope, each with a condition and a basis, confirmed by operations and clinical. The tab Dispositions Without Rules lists the buckets still uncovered."),
        ("How to read it", "NMTARA breakout",
         "The field encodes disposition bands, not individual acuity levels. A value such as BLS for NMTARA 1-5 (Override) describes a band and an override together, so a single digit read out of it mislabels the row."),
        ("Columns - Disposition Values", "Disposition value", "The value as it appears in the extract, trimmed of surrounding spaces."),
        ("Columns - Disposition Values", "Calls", "Calls in the window carrying that value."),
        ("Columns - NMTARA Bands", "Override band", "True when the band value contains the word override."),
        ("Columns - NMTARA Bands", "Band covers levels 1 to 5", "True when the band value spans levels one to five."),
        ("Columns - NMTARA Bands", "Band covers level 6", "True when the band value names level six."),
        ("Columns - Rule Reach", "Calls carrying that disposition", "The maximum number of calls the rule can move. The number that move depends on the condition."),
        ("Columns - Rule Reach", "Condition", "The test that must hold for the rule to apply to a call."),
        ("Columns - Dispositions Without Rules", "Disposition with no rule", "A disposition carried by calls in scope that no rule covers. These keep their current disposition in the recast.")]
data_info = pd.DataFrame(info, columns=["Section","Item","Detail"])

G = lambda n: RESULTS.get(n)
tabs = {
 "Data Info": data_info,
 "Disposition Values": G("disposition_values"),
 "Disposition by Market": G("disposition_by_market"),
 "NMTARA Bands": G("nmtara_bands"),
 "Rules and Reach": stack([("Rules defined", G("recast_rules")),
                           ("Rule conditions", G("rule_conditions")),
                           ("Population each rule applies to", G("rule_reach"))]),
 "Dispositions Without Rules": G("dispositions_without_rules")}

for f in os.listdir(OUT_DIR):
    if f.startswith("Nurse_Navigation_Disposition_Recast_") and f.endswith(".xlsx"):
        os.remove(os.path.join(OUT_DIR, f))

xlsx = os.path.join(OUT_DIR, f"Nurse_Navigation_Disposition_Recast_{RUN_ID}.xlsx")
with pd.ExcelWriter(xlsx, engine="xlsxwriter") as xl:
    for name, frame in tabs.items():
        if frame is None or not len(frame): continue
        frame.to_excel(xl, sheet_name=name[:31], index=False)
print("written:", xlsx)
print("tabs:", ", ".join(n for n, f in tabs.items() if f is not None and len(f)))